# Build Fashionpedia → Qdrant Index

Downloads images from Flickr, crops garments in memory, uploads to Supabase Storage, embeds via Gemini, upserts to Qdrant.

**Checkpoint/resume**: if Colab disconnects, reconnect and re-run from **Cell 3** onward — it picks up where it left off.

In [ ]:
!pip install -q requests pillow

from google.colab import drive
drive.mount('/content/drive')

import os
ANNOTATIONS_PATH = "/content/drive/MyDrive/fashionpedia_train.json"
assert os.path.exists(ANNOTATIONS_PATH), f"Upload fashionpedia_train.json to your Google Drive root first!"
print(f"Found annotations: {os.path.getsize(ANNOTATIONS_PATH) / 1e6:.0f} MB")

## 1. Install deps & mount Drive

In [ ]:
# Paste your keys here
QDRANT_URL = ""        # e.g. https://xxxxx.aws.cloud.qdrant.io:6333
QDRANT_API_KEY = ""
GEMINI_API_KEY = ""
SUPABASE_URL = ""      # e.g. https://xxxxx.supabase.co
SUPABASE_SERVICE_KEY = ""

assert all([QDRANT_URL, QDRANT_API_KEY, GEMINI_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_KEY]), "Fill in all keys above!"
print("Keys set ✓")

## 2. Paste your API keys

In [ ]:
import io
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
from PIL import Image

# --------------- Config ---------------

COLLECTION = "fashionpedia_v2"
EMBED_DIM = 768
EMBED_BATCH = 96
UPSERT_BATCH = 500
CHECKPOINT_EVERY = 500
GEMINI_EMBED_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents?key={GEMINI_API_KEY}"

SUPABASE_BUCKET = "crops"
SUPABASE_HEADERS = {
    "Authorization": f"Bearer {SUPABASE_SERVICE_KEY}",
    "apikey": SUPABASE_SERVICE_KEY,
}

WEARABLE_IDS = set(range(27))

FASHIONPEDIA_ATTRIBUTES = {
    0: "floral", 1: "graphic", 2: "striped", 3: "plain", 4: "lattice",
    5: "spotted", 6: "checked", 7: "solid color",
    8: "denim", 9: "chiffon", 10: "corduroy", 11: "cotton",
    12: "faux fur", 13: "knit", 14: "lace", 15: "leather",
    16: "linen", 17: "mesh", 18: "nylon", 19: "satin",
    20: "sequined", 21: "silk", 22: "suede", 23: "velvet", 24: "wool",
    25: "long sleeve", 26: "short sleeve", 27: "sleeveless",
    28: "maxi length", 29: "midi length", 30: "mini length",
    31: "crew neckline", 32: "v-neckline", 33: "turtleneck",
    34: "sweetheart neckline", 35: "straight fit", 36: "loose fit", 37: "tight fit",
}

CHECKPOINT_PATH = Path("/content/drive/MyDrive/closet_drift_checkpoint.json")
ITEMS_CACHE_PATH = Path("/content/drive/MyDrive/closet_drift_items_cache.json")

# --------------- Flickr URL upgrade ---------------

def upgrade_url(url):
    """Swap Flickr thumbnail suffixes (_n=320px, _m=500px) to _b (1024px)."""
    for s in ['_n.', '_m.']:
        if s in url:
            return url.replace(s, '_b.')
    return url

# --------------- Supabase Storage ---------------

def ensure_bucket():
    r = requests.get(f"{SUPABASE_URL}/storage/v1/bucket/{SUPABASE_BUCKET}", headers=SUPABASE_HEADERS, timeout=10)
    if r.status_code == 200:
        print(f"Supabase bucket '{SUPABASE_BUCKET}' exists")
        return
    resp = requests.post(
        f"{SUPABASE_URL}/storage/v1/bucket",
        headers={**SUPABASE_HEADERS, "Content-Type": "application/json"},
        json={"id": SUPABASE_BUCKET, "name": SUPABASE_BUCKET, "public": True},
        timeout=10,
    )
    resp.raise_for_status()
    print(f"Created Supabase bucket '{SUPABASE_BUCKET}'")

def upload_crop(filename, jpeg_bytes):
    resp = requests.post(
        f"{SUPABASE_URL}/storage/v1/object/{SUPABASE_BUCKET}/{filename}",
        headers={**SUPABASE_HEADERS, "Content-Type": "image/jpeg", "x-upsert": "true"},
        data=jpeg_bytes, timeout=30,
    )
    resp.raise_for_status()
    return f"{SUPABASE_URL}/storage/v1/object/public/{SUPABASE_BUCKET}/{filename}"

# --------------- Checkpoint ---------------

def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            data = json.loads(CHECKPOINT_PATH.read_text())
            n = data.get("items_done", 0)
            print(f"Resuming from checkpoint: {n} items already done")
            return n
        except Exception:
            pass
    return 0

def save_checkpoint(items_done):
    CHECKPOINT_PATH.write_text(json.dumps({"items_done": items_done, "collection": COLLECTION}))

# --------------- Gemini Embedding (sequential with retry) ---------------

def _embed_one_batch(texts, max_retries=5):
    body = {
        "requests": [
            {"model": "models/gemini-embedding-001", "content": {"parts": [{"text": t}]}, "outputDimensionality": EMBED_DIM}
            for t in texts
        ]
    }
    for attempt in range(max_retries):
        try:
            resp = requests.post(GEMINI_EMBED_URL, json=body, timeout=60)
            if resp.status_code == 429:
                wait = min(30 * (2 ** attempt), 120)
                print(f"    rate limited, waiting {wait}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return [e["values"] for e in resp.json()["embeddings"]]
        except requests.exceptions.HTTPError:
            raise
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(10)
                continue
            raise
    raise Exception(f"Failed after {max_retries} retries (rate limited)")

def embed_texts_sequential(texts):
    """Embed texts one batch at a time to avoid rate limits."""
    batches = [texts[i:i + EMBED_BATCH] for i in range(0, len(texts), EMBED_BATCH)]
    all_vecs = []
    for i, batch in enumerate(batches):
        try:
            vecs = _embed_one_batch(batch)
            all_vecs.extend(vecs)
        except Exception as e:
            print(f"  [embed] batch {i} failed permanently: {e}")
            all_vecs.extend([[0.0] * EMBED_DIM] * len(batch))
        # Small pause between batches to stay under rate limit
        if i < len(batches) - 1:
            time.sleep(1)
    return all_vecs

# --------------- Qdrant ---------------

def ensure_collection():
    r = requests.get(f"{QDRANT_URL}/collections/{COLLECTION}", headers={"api-key": QDRANT_API_KEY}, timeout=10)
    if r.status_code == 200:
        print(f"Collection '{COLLECTION}' exists")
        return
    resp = requests.put(
        f"{QDRANT_URL}/collections/{COLLECTION}",
        headers={"api-key": QDRANT_API_KEY, "Content-Type": "application/json"},
        json={"vectors": {"size": EMBED_DIM, "distance": "Cosine"}, "optimizers_config": {"indexing_threshold": 0}},
        timeout=10,
    )
    resp.raise_for_status()
    print(f"Created collection '{COLLECTION}'")

def upsert_batch(points):
    resp = requests.put(
        f"{QDRANT_URL}/collections/{COLLECTION}/points",
        headers={"api-key": QDRANT_API_KEY, "Content-Type": "application/json"},
        json={"points": points}, timeout=60,
    )
    resp.raise_for_status()

# --------------- Image Download + Crop ---------------

def download_image(original_url):
    if not original_url:
        return None
    try:
        resp = requests.get(original_url, timeout=30)
        resp.raise_for_status()
        return Image.open(io.BytesIO(resp.content))
    except Exception:
        return None

def crop_and_upload(img, bbox, ann_id, ann_img_w, ann_img_h, padding=0.1):
    crop_filename = f"{ann_id}.jpg"

    actual_w, actual_h = img.size
    scale_x = actual_w / ann_img_w if ann_img_w else 1
    scale_y = actual_h / ann_img_h if ann_img_h else 1

    x = bbox[0] * scale_x
    y = bbox[1] * scale_y
    w = bbox[2] * scale_x
    h = bbox[3] * scale_y

    pad_x, pad_y = w * padding, h * padding
    left = max(0, int(x - pad_x))
    top = max(0, int(y - pad_y))
    right = min(actual_w, int(x + w + pad_x))
    bottom = min(actual_h, int(y + h + pad_y))

    if right - left < 10 or bottom - top < 10:
        return None

    crop = img.crop((left, top, right, bottom))
    max_side = max(crop.size)
    if max_side > 400:
        scale = 400 / max_side
        crop = crop.resize((int(crop.size[0] * scale), int(crop.size[1] * scale)), Image.LANCZOS)

    crop_w, crop_h = crop.size

    buf = io.BytesIO()
    crop.save(buf, "JPEG", quality=85)
    try:
        url = upload_crop(crop_filename, buf.getvalue())
        return url, crop_w, crop_h
    except Exception as e:
        print(f"  [upload] {crop_filename} failed: {e}")
        return None

# --------------- Build Items ---------------

def build_items():
    if ITEMS_CACHE_PATH.exists():
        print(f"Loading cached items from Drive...")
        with open(ITEMS_CACHE_PATH) as f:
            items = json.load(f)
        print(f"Loaded {len(items)} items from cache")
        return items

    print(f"Loading annotations...")
    with open(ANNOTATIONS_PATH) as f:
        data = json.load(f)

    categories = {c["id"]: c["name"] for c in data["categories"]}
    attributes = {a["id"]: a["name"] for a in data["attributes"]}
    images_map = {img["id"]: img for img in data["images"]}

    img_anns = {}
    for ann in data["annotations"]:
        if ann["category_id"] in WEARABLE_IDS and ann.get("bbox"):
            img_anns.setdefault(ann["image_id"], []).append(ann)

    total_images = len(img_anns)
    print(f"Processing {total_images} images...")

    items = []
    skipped = 0
    processed = 0
    uploaded = 0
    image_ids = list(img_anns.keys())
    dl_batch = 20
    t0 = time.time()

    for batch_start in range(0, len(image_ids), dl_batch):
        batch_ids = image_ids[batch_start : batch_start + dl_batch]

        downloaded = {}
        with ThreadPoolExecutor(max_workers=8) as pool:
            futures = {}
            for img_id in batch_ids:
                img_info = images_map.get(img_id, {})
                url = upgrade_url(img_info.get("original_url", ""))
                if url:
                    futures[pool.submit(download_image, url)] = img_id
            for future in as_completed(futures):
                img_id = futures[future]
                result = future.result()
                if result:
                    downloaded[img_id] = result

        for img_id in batch_ids:
            img = downloaded.get(img_id)
            if not img:
                skipped += len(img_anns[img_id])
                continue

            img_info = images_map[img_id]
            ann_img_w = img_info.get("width", img.size[0])
            ann_img_h = img_info.get("height", img.size[1])

            for ann in img_anns[img_id]:
                crop_result = crop_and_upload(img, ann["bbox"], ann["id"], ann_img_w, ann_img_h)
                if not crop_result:
                    skipped += 1
                    continue
                crop_url, crop_w, crop_h = crop_result
                uploaded += 1
                cat_id = ann["category_id"]
                cat_name = categories.get(cat_id, "clothing")
                attr_ids = ann.get("attribute_ids", [])
                attr_names = [attributes.get(a, FASHIONPEDIA_ATTRIBUTES.get(a, "")) for a in attr_ids]
                attr_names = [a for a in attr_names if a]
                desc_parts = []
                if attr_names:
                    desc_parts.append(", ".join(attr_names))
                desc_parts.append(cat_name)
                items.append({
                    "id": len(items),
                    "description": " ".join(desc_parts),
                    "category": cat_name,
                    "category_id": cat_id,
                    "attributes": attr_names,
                    "crop_url": crop_url,
                    "width": crop_w,
                    "height": crop_h,
                })

            img.close()
            processed += 1

        if processed % 20 == 0 and processed > 0:
            elapsed = time.time() - t0
            rate = processed / elapsed if elapsed > 0 else 0
            eta = (total_images - processed) / rate if rate > 0 else 0
            print(f"  [{processed}/{total_images}] {uploaded} crops | {skipped} skipped | {rate:.1f} img/s | ETA {eta/60:.0f}m")

    print(f"Built {len(items)} items ({skipped} skipped) in {(time.time()-t0)/60:.1f}m")

    print(f"Saving items cache to Drive...")
    with open(ITEMS_CACHE_PATH, "w") as f:
        json.dump(items, f)

    return items

# --------------- Run it ---------------

print("=== Phase 0: Supabase bucket ===")
ensure_bucket()

print("\n=== Phase 1: Download + Crop + Upload ===")
items = build_items()
assert items, "No items built!"
print(f"\n{len(items)} items ready")

print("\n=== Phase 2: Qdrant collection ===")
ensure_collection()

print("\n=== Phase 3: Embed + Upsert ===")
start_from = load_checkpoint()
total = len(items)

if start_from >= total:
    print(f"All {total} items already indexed!")
else:
    if start_from > 0:
        print(f"Skipping first {start_from} items (already done)")

    remaining = items[start_from:]
    done = start_from
    t0 = time.time()

    for chunk_start in range(0, len(remaining), UPSERT_BATCH):
        chunk = remaining[chunk_start : chunk_start + UPSERT_BATCH]
        texts = [item["description"] for item in chunk]

        try:
            vectors = embed_texts_sequential(texts)
        except Exception as e:
            print(f"FATAL embed error at item {done}: {e}")
            save_checkpoint(done)
            print(f"Checkpoint saved at {done}. Re-run to resume.")
            raise

        points = []
        for item, vector in zip(chunk, vectors):
            points.append({
                "id": item["id"],
                "vector": vector,
                "payload": {
                    "description": item["description"],
                    "category": item["category"],
                    "category_id": item["category_id"],
                    "attributes": item["attributes"],
                    "crop_url": item["crop_url"],
                    "width": item["width"],
                    "height": item["height"],
                },
            })

        try:
            upsert_batch(points)
        except Exception as e:
            print(f"Upsert error at item {done}: {e}")
            save_checkpoint(done)
            print(f"Checkpoint saved at {done}. Re-run to resume.")
            raise

        done += len(chunk)

        if done % CHECKPOINT_EVERY < UPSERT_BATCH or chunk_start + UPSERT_BATCH >= len(remaining):
            save_checkpoint(done)

        elapsed = time.time() - t0
        rate = (done - start_from) / elapsed if elapsed > 0 else 0
        eta = (total - done) / rate if rate > 0 else 0
        print(f"  [{done}/{total}] embedded+upserted | {rate:.0f} items/s | ETA {eta/60:.0f}m")

    print(f"\nDone! {total} items indexed in {(time.time()-t0)/60:.1f}m")
    CHECKPOINT_PATH.unlink(missing_ok=True)

## 3. Run the build

This is the main cell. If Colab disconnects, reconnect, re-run cells 1 & 2, then re-run this — it resumes from checkpoint.